In [ ]:
# parse_instruction_ver17 (target 精度改善)

## 背景
- ver15/ver16 では action 精度は 100% に近いが、target が `subject` / `person` のままになるケースが多く後段タスクで使えない。
- instruction に「何を対象に編集するか」が明示されているにもかかわらず、抽出できていない。

## 目的
- instruction から target を直接抽出できる action の範囲を拡大し、target_score と total の改善を目指す。

## 制約
- class / subclass 使用禁止
- 入力は instruction 文字列のみ

## 試行モデル
- `v17a` : v15d ベースライン（単体 target accuracy の診断）
- `v17b` : instruction から target を直接抽出する範囲を全 action に拡大
- `v17c` : v17b + retrieval fallback（target のみ。params は v15d ロジック維持）
- `v17d` : v17c を primary にした multi-task 版（v16d の aux strategy 組み合わせ）

2


In [1]:
from __future__ import annotations

import copy
import json
import re
import sys
from collections import defaultdict
from difflib import SequenceMatcher
from pathlib import Path

WORKSPACE = Path('/workspace')
SRC_DIR = WORKSPACE / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from parse.data_loading import load_base_records, load_grouped_unknown_records

DATA_DIR     = WORKSPACE / 'data'
NOTEBOOK_DIR = WORKSPACE / 'notebook'
OUTPUT_DIR   = NOTEBOOK_DIR / 'ver17_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH      = DATA_DIR / 'annotations.jsonl'
GT_PATH       = DATA_DIR / 'annotations_gt_task_ver09.json'
GROUPED_PATHS = [
    DATA_DIR / 'annotations_grouped_ver01.json',
    DATA_DIR / 'annotations_grouped_ver02.json',
]

base_records    = load_base_records(RAW_PATH, GT_PATH)
unknown_records = load_grouped_unknown_records(
    GROUPED_PATHS, base_records, instruction_keys=('ver2', 'ver3', 'ver4')
)
print('base_records:', len(base_records))
print('unknown_records:', len(unknown_records))

base_records: 100
unknown_records: 600


## 1. ユーティリティ + 評価関数（v15d ロジック再実装）

In [12]:
# ---- text utilities ----

def normalize_text(x):
    if x is None: return ''
    t = str(x).lower().replace('_', ' ').strip()
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def flatten_json(v, prefix=''):
    out = {}
    if isinstance(v, dict):
        for k, c in v.items():
            p = f'{prefix}.{k}' if prefix else str(k)
            out.update(flatten_json(c, p))
    elif isinstance(v, list):
        for i, c in enumerate(v):
            out.update(flatten_json(c, f'{prefix}[{i}]'))
    else:
        out[prefix] = normalize_text(v)
    return out

def text_similarity(a, b):
    aa, bb = normalize_text(a), normalize_text(b)
    if not aa and not bb: return 1.0
    if not aa or not bb:  return 0.0
    j = len(set(aa.split()) & set(bb.split())) / max(1, len(set(aa.split()) | set(bb.split())))
    return 0.6 * j + 0.4 * SequenceMatcher(None, aa, bb).ratio()

# ---- single-task scorer (ver15 / evaluate_records style) ----

def score_primary(pred, gt_primary):
    task = pred.get('tasks', [{}])[0]
    action = 1.0 if task.get('action', '') == gt_primary.get('action', '') else 0.0
    pt = task.get('target', '')
    gt = gt_primary.get('target', '')
    pt = ' '.join(normalize_text(x) for x in pt) if isinstance(pt, list) else normalize_text(pt)
    gt = ' '.join(normalize_text(x) for x in gt) if isinstance(gt, list) else normalize_text(gt)
    target = 1.0 if (pt and gt and (pt in gt or gt in pt)) else 0.0
    pp = flatten_json(task.get('params', {}))
    gp = flatten_json(gt_primary.get('params', {}))
    if not gp:   params = 1.0
    elif not pp: params = 0.0
    else:
        m = sum(1 for k, gv in gp.items() if k in pp and (pp[k] == gv or pp[k] in gv or gv in pp[k]))
        params = m / len(gp)
    total = 0.5 * action + 0.2 * target + 0.3 * params
    return {'action_score': round(action,4), 'target_score': round(target,4),
            'params_score': round(params,4), 'total': round(total,4)}

def evaluate_records(records, predict_fn):
    rows, pred_map, dbg_map = [], {}, {}
    for r in records:
        pred, dbg = predict_fn(r)
        key = r['prediction_key']
        pred_map[key] = pred
        dbg_map[key]  = dbg
        s = score_primary(pred, r['gt_primary'])
        rows.append({'prediction_key': key, 'video_path': r['video_path'], **s})
    overall = {k: round(sum(x[k] for x in rows)/len(rows), 4)
               for k in ['action_score','target_score','params_score','total']}
    return {'rows': rows, 'overall': overall, 'predictions': pred_map, 'debug': dbg_map}

# ---- multi-task scorer: ver16 準拠 (soft avg, task_score includes params) ----

def task_score_multi(pred_task, gt_task):
    """ver16 の task_score と同一: action(0.5) + target(0.2) + params(0.3)"""
    action = 1.0 if pred_task.get('action', '') == gt_task.get('action', '') else 0.0
    pt = normalize_text(' '.join(pred_task['target']) if isinstance(pred_task.get('target'), list) else pred_task.get('target', ''))
    gt = normalize_text(' '.join(gt_task['target']) if isinstance(gt_task.get('target'), list) else gt_task.get('target', ''))
    target = 1.0 if (pt and gt and (pt in gt or gt in pt)) else 0.0
    pp = flatten_json(pred_task.get('params', {}))
    gp = flatten_json(gt_task.get('params', {}))
    if not gp:   params = 1.0
    elif not pp: params = 0.0
    else:
        m = sum(1 for k, gv in gp.items() if k in pp and (pp[k] == gv or pp[k] in gv or gv in pp[k]))
        params = m / len(gp)
    return 0.5 * action + 0.2 * target + 0.3 * params

def score_multi(pred_tasks, gt_tasks):
    """ver16 の multi_task_score と同一: soft coverage + soft precision (weights: 0.55/0.35/0.10)"""
    pred_tasks = pred_tasks or []
    gt_tasks   = gt_tasks   or []
    if not pred_tasks and not gt_tasks:
        return {'coverage': 1.0, 'precision': 1.0, 'count_alignment': 1.0, 'total': 1.0}
    coverage  = sum(max((task_score_multi(p, g) for p in pred_tasks), default=0.0) for g in gt_tasks) / max(1, len(gt_tasks))
    precision = sum(max((task_score_multi(p, g) for g in gt_tasks), default=0.0) for p in pred_tasks) / max(1, len(pred_tasks))
    m = max(1, len(pred_tasks), len(gt_tasks))
    count_alignment = 1.0 - abs(len(pred_tasks) - len(gt_tasks)) / m
    total = 0.55 * coverage + 0.35 * precision + 0.10 * count_alignment
    return {'coverage': round(coverage,4), 'precision': round(precision,4),
            'count_alignment': round(count_alignment,4), 'total': round(total,4)}

def evaluate_multi(records, predict_fn):
    rows, pred_map = [], {}
    for r in records:
        pred = predict_fn(r)
        key  = r['prediction_key']
        pred_map[key] = pred
        s = score_multi(pred['tasks'], r['gt_tasks'])
        rows.append({'prediction_key': key, 'video_path': r['video_path'], **s})
    overall = {k: round(sum(x[k] for x in rows)/len(rows), 4)
               for k in ['coverage','precision','count_alignment','total']}
    return {'rows': rows, 'overall': overall, 'predictions': pred_map}

print('utilities ready (multi-task scorer = ver16 準拠)')

utilities ready (multi-task scorer = ver16 準拠)


## 2. action 推論 + v15d target/params ロジック（共通部品）
- v15d のロジックをそのまま再実装し、v17 系の比較基準とする。

In [13]:
_STYLE_VOCAB = [
    'anime', 'cyberpunk', 'pixel art', 'ukiyo', 'ghibli', 'watercolor',
    'oil painting', 'comic style', 'woodblock', '16-bit', 'retro pixel',
    'aesthetic', 'painting style', 'art style', 'american comic',
]
_COLOR_WORDS = [
    'violet', 'blue', 'red', 'green', 'navy', 'emerald', 'burgundy', 'magenta',
    'orange', 'purple', 'pink', 'yellow', 'black', 'white', 'gray', 'grey',
    'silver', 'gold', 'teal', 'cyan', 'indigo', 'maroon', 'beige', 'coral',
    'turquoise', 'aqua', 'lime', 'lavender', 'brown', 'crimson', 'amber',
]

def infer_action(inst):
    t = inst.lower()
    if re.search(r'\barc\s+shot\b|\borbit\b|revolving\s+around', t): return 'orbit_camera'
    if re.search(r'increase\s+the\s+amount\s+of', t): return 'increase_amount'
    if re.search(r'increase\s+the\s+number\s+of', t): return 'add_object'
    if re.search(r'\bdolly[- ]?in\b', t): return 'dolly_in'
    if re.search(r'\bzoom[- ]?out\b', t): return 'zoom_out'
    if re.search(r'\bzoom[- ]?in\b', t):  return 'zoom_in'
    if re.search(r'low[- ]?angle|high[- ]?angle', t): return 'change_camera_angle'
    if re.search(r'(?:camera|shot|perspective|view)\s+(?:to|into)\s+(?:a\s+)?(?:low|high)', t): return 'change_camera_angle'
    if (bool(re.search(r'transform\s+(?:the\s+)?(?:entire\s+)?(?:video|scene|frame)', t))
            or re.search(r'apply\s+(?:a\s+)?(?:\w+\s+)*(?:art\s+)?style', t)):
        if any(w in t for w in _STYLE_VOCAB): return 'apply_style'
    if any(w in t for w in _STYLE_VOCAB): return 'apply_style'
    if re.search(r'\breplace\b', t):
        if re.search(r'replace\s+(?:the\s+)?(?:(?:\w+\s+){0,6})background', t): return 'replace_background'
        if 'with' in t: return 'replace_object'
    if re.search(r'\bremove\b', t): return 'remove_object'
    if re.search(r'(?:add|apply|enhance).*(?:glow(?:ing)?|aura|flame|lightning|sparkle|decoration\s+effect|stage\s+lighting)', t): return 'add_effect'
    if re.search(r'(?:glowing|neon\s+electric|neon\s+glow|electric\s+glow)', t): return 'add_effect'
    if re.search(r'\b(?:add|insert|place|introduce)\b', t): return 'add_object'
    if re.search(r'(?:change|modify)\s+(?:the\s+)?(?:\w+\s+)*color', t): return 'change_color'
    if re.search(r'color\s+(?:of|to)\s+', t) and re.search(r'\b(?:change|modify|transform)\b', t): return 'change_color'
    return 'edit_motion'

def _extract_new_color(inst):
    t = inst.lower()
    m = re.search(r'\bto\s+(?:a\s+)?(?:(?:vibrant|bright|deep|dark|solid|metallic|neon|bold|glossy){0,2}\s*)'
                  r'([a-z]+(?:\s+[a-z]+){0,2})\s*(?:color|hue|throughout|while|[,.]|$)', t)
    if m:
        val = m.group(1).strip()
        if val not in {'color', 'hue', 'a', 'an', 'the', 'vibrant', 'metallic'}: return val
    return ''

def _extract_color_words(inst):
    t = inst.lower()
    found = []
    for pat in [r'(?:vibrant\s+)?metallic\s+[a-z]+', r'(?:deep|bright|dark|solid)\s+[a-z]+(?:\s+[a-z]+)?',
                r'neon\s+[a-z]+', r'navy\s+blue', r'emerald\s+green']:
        for m in re.finditer(pat, t): found.append(m.group())
    for cw in _COLOR_WORDS:
        if re.search(r'\b' + cw + r'\b', t) and not any(cw in f for f in found):
            found.append(cw)
    return found

def default_target_v15d(action):
    return {
        'replace_background': 'background', 'replace_object': 'object',
        'remove_object': 'object', 'add_object': 'object',
        'increase_amount': 'object', 'change_color': 'subject',
        'apply_style': 'full_frame', 'zoom_out': 'camera_view',
        'zoom_in': 'camera_view', 'dolly_in': 'subject',
        'change_camera_angle': 'subject', 'orbit_camera': 'subject',
        'edit_motion': 'person', 'add_effect': 'subject',
    }.get(action, 'subject')

def default_params_v15d(action, inst):
    t = inst.lower(); p = {}
    if action == 'apply_style':
        for pat, val in [('ukiyo', 'ukiyo-e'), ('ghibli', 'ghibli'), ('watercolor', 'watercolor'),
                         ('oil painting', 'oil_painting'), (r'comic style|american comic|comic book', 'american_comic_style'),
                         (r'\bpixel\b|16-bit', 'pixel_art'), (r'\banime\b', 'anime'), ('cyberpunk', 'cyberpunk')]:
            if re.search(pat, t): p['style'] = val; break
    elif action in ('zoom_in', 'zoom_out', 'dolly_in'):
        p['motion_type'] = action
        if re.search(r'\b(?:slow|smooth|gradual|subtle|steady)\b', t): p['speed'] = 'gradual'
    elif action == 'change_camera_angle':
        if re.search(r'low[- ]?angle', t): p['angle'] = 'low_angle'
        elif re.search(r'high[- ]?angle', t): p['angle'] = 'high_angle'
    elif action == 'orbit_camera': p['trajectory'] = 'arc'
    elif action == 'add_effect': p['effect_type'] = 'glow_or_decoration'
    return p

def extract_target_params_v15d(action, inst):
    """ver16 の extract_primary_target_params と同一実装。
    extraction_actions + change_camera_angle の regex 抽出を含む。"""
    target = default_target_v15d(action)
    params = {}
    t = inst.lower()
    if action == 'change_color':
        m = re.search(r'(?:change|modify)\s+(?:the\s+)?(?:\w+\s+)?color\s+of\s+(?:the\s+)?(.+?)\s+to\s+', inst, re.I)
        if not m: m = re.search(r'(?:change|modify)\s+the\s+(.+?)\s+to\s+', inst, re.I)
        if m: target = m.group(1).strip()
        nc = _extract_new_color(inst); colors = _extract_color_words(inst)
        if nc:
            params['new_color'] = nc
            params['mentioned_colors'] = [nc] + [c for c in colors if c != nc and c not in nc][:5]
    elif action == 'replace_object':
        m = re.search(r'replace\s+(?:the\s+)?(.+?)\s+with\s+', inst, re.I)
        if m: target = m.group(1).strip()
        m2 = re.search(r'\bwith\s+(?:a\s+|an\s+)?(?:\w+\s+){0,3}(.+?)(?:\s+(?:throughout|while)|[,.]|$)', inst, re.I)
        params = {'replacement': {'category': m2.group(1).strip().split()[-1].lower() if m2 else 'replacement', 'viewpoint': 'match_source'}}
    elif action == 'replace_background':
        target = 'background'
        m = re.search(r'\bwith\s+(?:a\s+)?(.+?)(?:\s+featuring|[,.]|$)', inst, re.I)
        params = {'new_scene': {'style': re.sub(r'\s+', '_', m.group(1).strip()[:40].lower())} if m else {}}
    elif action == 'add_object':
        for pat in [r'increase\s+the\s+number\s+of\s+(.+?)\s+(?:in|by|to|on|with)',
                    r'(?:add|insert)\s+(?:a|an|another|additional)\s+(.+?)(?:\s+(?:to|in|on)|[,.]|$)',
                    r'(?:place|introduce)\s+(?:a|an)\s+(.+?)(?:\s+(?:on|in)|[,.]|$)',
                    r'adding\s+(?:a\s+second\s+|more\s+)?(.+?)(?:\s+(?:to|in|on)|[,.]|$)']:
            m = re.search(pat, inst, re.I)
            if m: target = m.group(1).strip(); break
    elif action == 'remove_object':
        m = re.search(r'remove\s+(?:the\s+)?(.+?)(?:\s+(?:from|and)|[,.]|$)', inst, re.I)
        if m: target = m.group(1).strip()
    elif action == 'increase_amount':
        m = re.search(r'increase\s+(?:the\s+)?(?:amount|number)\s+of\s+(.+?)(?:\s+(?:on|in|to)|[,.]|$)', inst, re.I)
        if m: target = m.group(1).strip()
        params = {'density': 'dense'}
    else:
        # ver16 の else ブランチ: camera/style/motion に smart defaults + change_camera_angle 抽出
        params = default_params_v15d(action, inst)
        if action == 'apply_style':   target = 'full_frame'
        elif action in ('zoom_in', 'zoom_out'): target = 'camera_view'
        elif action == 'change_camera_angle':
            m = re.search(r'(?:toward|towards|on|looking\s+(?:up|down)\s+at)\s+(?:the\s+)?([a-z][a-z\s]{1,25}?)(?:\s*[,;.]|$)', t)
            if m: target = m.group(1).strip()
    return target, params

_EXTRACTION_ACTIONS = {'change_color','replace_object','replace_background',
                       'add_object','remove_object','increase_amount'}

def nearest_same_action(inst, action, pool, skip_video):
    best, best_s = None, -1.0
    for c in pool:
        if c['video_path'] == skip_video: continue
        if c['gt_primary']['action'] != action: continue
        s = text_similarity(inst, c['instruction'])
        if s > best_s: best_s, best = s, c
    return best, best_s

def predict_primary_v15d(inst, pool, skip_video):
    """ver16 の predict_primary_task と同一 (near_for_aux は内部のみ)。"""
    action = infer_action(inst)
    target, params = extract_target_params_v15d(action, inst)

    if action not in _EXTRACTION_ACTIONS:
        near, sim = nearest_same_action(inst, action, pool, skip_video)
        if near is not None and sim > 0.15:
            gt = near['gt_primary']
            target = gt.get('target', target)
            merged = copy.deepcopy(gt.get('params', {}))
            for k, v in params.items(): merged[k] = v
            params = merged
        if action == 'apply_style':   target = 'full_frame'
        elif action in ('zoom_in', 'zoom_out'): target = 'camera_view'
        elif action == 'orbit_camera': params['trajectory'] = 'arc'
        elif action == 'edit_motion' and target in ('subject', 'full_frame', 'camera_view', 'face'): target = 'person'
        elif action == 'add_effect': params.setdefault('effect_type', 'glow_or_decoration')

    task = {'action': action, 'target': target, 'constraints': [], 'params': params}
    return task

print('v15d/v16d core ready (ver16 extract_primary_target_params 準拠)')

v15d/v16d core ready (ver16 extract_primary_target_params 準拠)


## 3. v17a — v15d ベースラインと target エラー診断
- 意図: ver16 での target=subject 問題がどの action/instruction パターンで発生しているかを定量化する。
- 目的: action 別 target_score と「subject のまま」の件数を把握し、改善優先度を決める。
- 判定基準: action 別 target_score が 0.5 未満の group を特定する。

In [15]:
def predict_v17a(record):
    task = predict_primary_v15d(record['instruction'], base_records, record['video_path'])
    return {'tasks': [task]}, {'version': 'v17a'}

res_v17a = evaluate_records(base_records, predict_v17a)
print('v17a (v15d baseline):', res_v17a['overall'])

# ---- action 別 target_score + target=subject 件数 ----
by_action_tgt = defaultdict(list)
subject_default_rows = []

for row in res_v17a['rows']:
    key = row['prediction_key']
    rec = next(r for r in base_records if r['prediction_key'] == key)
    act = rec['gt_primary']['action']
    by_action_tgt[act].append(row['target_score'])
    pred_tgt = res_v17a['predictions'][key]['tasks'][0].get('target', '')
    gt_tgt   = rec['gt_primary'].get('target', '')
    if pred_tgt in ('subject', 'person') and pred_tgt != normalize_text(str(gt_tgt)):
        subject_default_rows.append({
            'action': act, 'instruction': rec['instruction'],
            'gt_target': gt_tgt, 'pred_target': pred_tgt
        })

print('\n--- action 別 target_score (昇順) ---')
for act in sorted(by_action_tgt, key=lambda a: sum(by_action_tgt[a])/len(by_action_tgt[a])):
    sc = by_action_tgt[act]
    print(f'  {act:25s} n={len(sc):2d}  target_avg={sum(sc)/len(sc):.3f}')

print(f'\n合計 target=subject/person のまま誤り: {len(subject_default_rows)} 件')
print('\n--- target=subject 誤り の instruction サンプル (先頭 10件) ---')
for row in subject_default_rows[:10]:
    print(f'  [{row["action"]:22s}] GT: {str(row["gt_target"])[:35]:35s} | {row["instruction"][:70]}')

v17a (v15d baseline): {'action_score': 1.0, 'target_score': 0.69, 'params_score': 0.5623, 'total': 0.8067}

--- action 別 target_score (昇順) ---
  dolly_in                  n= 3  target_avg=0.000
  add_effect                n= 3  target_avg=0.000
  orbit_camera              n= 1  target_avg=0.000
  change_camera_angle       n=10  target_avg=0.100
  add_object                n=11  target_avg=0.273
  zoom_in                   n=11  target_avg=0.545
  change_color              n=11  target_avg=0.909
  edit_motion               n=12  target_avg=0.917
  replace_background        n=12  target_avg=1.000
  apply_style               n=15  target_avg=1.000
  replace_object            n= 6  target_avg=1.000
  remove_object             n= 2  target_avg=1.000
  increase_amount           n= 1  target_avg=1.000
  zoom_out                  n= 2  target_avg=1.000

合計 target=subject/person のまま誤り: 3 件

--- target=subject 誤り の instruction サンプル (先頭 10件) ---
  [dolly_in              ] GT: man's face          

## 4. v17b — target 直接抽出を全 action に拡大
- 意図: v15d は extraction_actions（6種）のみ instruction からの target 抽出をしていた。残りの action は retrieval か default に頼っていた。
- 目的: `dolly_in` / `change_camera_angle` / `orbit_camera` / `add_effect` / `edit_motion` にも instruction 抽出を追加し、target=subject 誤りを減らす。
- 判定基準: v17a と比べて target_score が改善するか。

In [16]:
# ---- 追加 target 抽出ルール ----

_PERSON_WORDS = [
    'man', 'woman', 'person', 'child', 'baby', 'girl', 'boy', 'figure',
    'player', 'presenter', 'chef', 'dancer', 'athlete', 'model', 'actor',
    'performer', 'musician', 'teacher', 'student', 'worker', 'character',
]

def extract_noun_subject(inst):
    """instruction から主語となる名詞句を抽出する汎用ルール。マッチしなければ None を返す。"""
    t = inst.lower()
    # [1] 「toward/on/around/focusing on the [X]」
    m = re.search(
        r'(?:toward|towards|around|on|of|upon|at)\s+the\s+'
        r'([a-z][a-z\s]{1,35}?)'
        r'(?:\s*[,;.]|\s+(?:in|on|at|to|with|while|from|that|which|and)|$)',
        t)
    if m:
        val = m.group(1).strip()
        if len(val.split()) <= 5 and val not in ('video', 'scene', 'frame', 'background', 'foreground'):
            return val
    # [2] 「the [noun] in the foreground/center/scene」
    m = re.search(r'the\s+(\w+(?:\s+\w+)?)\s+in\s+the\s+(?:foreground|center|scene|shot|background)', t)
    if m: return m.group(1).strip()
    # [3] 「make/have the [X] ...」
    m = re.search(r'(?:make|have|get|let)\s+the\s+([a-z][a-z\s]{1,30}?)\s+(?:wave|nod|spin|jump|walk|run|dance|move|perform|do|look)', t)
    if m: return m.group(1).strip()
    # [4] 「apply [effect] to the [X]」
    m = re.search(r'(?:apply|add|place)\s+.{1,30}?\s+(?:to|on|around)\s+the\s+([a-z][a-z\s]{1,30}?)(?:\s*[,;.]|\s+|$)', t)
    if m:
        val = m.group(1).strip()
        if val not in ('video', 'scene', 'frame'): return val
    # [5] 既知の人物単語
    for pw in _PERSON_WORDS:
        if re.search(r'\b' + pw + r'\b', t): return pw
    return None


def extract_target_v17b(action, inst):
    """extraction_actions は v15d と同じ。それ以外の action にも instruction 抽出を追加。"""
    t = inst.lower()
    target, params = extract_target_params_v15d(action, inst)
    if action in _EXTRACTION_ACTIONS:
        return target, params

    params = default_params_v15d(action, inst)

    if action == 'apply_style':
        return 'full_frame', params

    if action in ('zoom_in', 'zoom_out'):
        return 'camera_view', params

    if action == 'dolly_in':
        m = re.search(r'dolly[- ]?in\s+(?:to\s+|toward\s+)?(?:the\s+)?([a-z][a-z\s]{1,30}?)'
                      r'(?:\s*[,;.]|\s+(?:in|on|at|to|while|from)|$)', t)
        if m:
            val = m.group(1).strip()
            if val not in ('video', 'scene', 'frame'): return val, params
        noun = extract_noun_subject(inst)
        return (noun if noun else 'subject'), params

    if action == 'change_camera_angle':
        m = re.search(r'(?:toward|towards|at|on|facing)\s+(?:the\s+)?([a-z][a-z\s]{1,30}?)'
                      r'(?:\s*[,;.]|\s+(?:in|on|at|to|with|while|from|that|and)|$)', t)
        if m:
            val = m.group(1).strip()
            if val not in ('video', 'scene', 'frame', 'camera', 'background', 'angle'):
                return val, params
        noun = extract_noun_subject(inst)
        return (noun if noun else 'subject'), params

    if action == 'orbit_camera':
        m = re.search(r'(?:orbit|arc|revolv\w+|circl\w+)\s+(?:around\s+)?(?:the\s+)?([a-z][a-z\s]{1,30}?)'
                      r'(?:\s*[,;.]|\s+(?:in|on|at|to|with|while|from|that|and)|$)', t)
        if m:
            val = m.group(1).strip()
            if val not in ('video', 'scene', 'frame'): return val, params
        noun = extract_noun_subject(inst)
        return (noun if noun else 'subject'), params

    if action == 'add_effect':
        m = re.search(r'(?:glow(?:ing)?|aura|flame|sparkle|lightning|neon)[\w\s]*?'
                      r'(?:to|on|around|for)\s+(?:the\s+)?([a-z][a-z\s]{1,30}?)'
                      r'(?:\s*[,;.]|\s+(?:in|on|at|to|with|while|from|that|and)|$)', t)
        if m:
            val = m.group(1).strip()
            if val not in ('video', 'scene', 'frame'): return val, params
        m = re.search(r'(?:add|apply|enhance)\s+(?:\w+\s+){0,4}(?:to|on|around)\s+(?:the\s+)?([a-z][a-z\s]{1,30}?)'
                      r'(?:\s*[,;.]|\s+(?:in|on|at|to|with|while|from|that|and)|$)', t)
        if m:
            val = m.group(1).strip()
            if val not in ('video', 'scene', 'frame'): return val, params
        noun = extract_noun_subject(inst)
        return (noun if noun else 'subject'), params

    if action == 'edit_motion':
        m = re.search(
            r"(?:make|have|get|let)\s+the\s+([a-z][a-z\s']{1,30}?)\s+"
            r'(?:wave|nod|spin|jump|walk|run|dance|move|perform|do|look|turn|bow)',
            t)
        if m: return m.group(1).strip(), params
        m = re.search(r"([a-z\s']{1,30}?)'s\s+(?:hand|arm|head|body|leg|motion|gesture|movement)", t)
        if m: return m.group(1).strip(), params
        noun = extract_noun_subject(inst)
        return (noun if noun else 'person'), params

    return default_target_v15d(action), params


def predict_v17b(record):
    inst = record['instruction']
    action = infer_action(inst)
    target, params = extract_target_v17b(action, inst)
    task = {'action': action, 'target': target, 'constraints': [], 'params': params}
    return {'tasks': [task]}, {'version': 'v17b', 'action': action, 'target': target}

res_v17b = evaluate_records(base_records, predict_v17b)
print('v17b (expanded target extraction):', res_v17b['overall'])

print('\nv17a -> v17b delta:')
for k in ['action_score','target_score','params_score','total']:
    delta = res_v17b['overall'][k] - res_v17a['overall'][k]
    print(f'  {k:15s}: {res_v17a["overall"][k]:.4f} -> {res_v17b["overall"][k]:.4f}  ({delta:+.4f})')

v17b (expanded target extraction): {'action_score': 1.0, 'target_score': 0.66, 'params_score': 0.5489, 'total': 0.7967}

v17a -> v17b delta:
  action_score   : 1.0000 -> 1.0000  (+0.0000)
  target_score   : 0.6900 -> 0.6600  (-0.0300)
  params_score   : 0.5623 -> 0.5489  (-0.0134)
  total          : 0.8067 -> 0.7967  (-0.0100)


### v17b action 別詳細
- どの action で改善・悪化があったかを確認する。

In [17]:
def per_action_scores(res, records):
    ba = defaultdict(lambda: {'total':[], 'target':[], 'params':[]})
    for row in res['rows']:
        key = row['prediction_key']
        rec = next(r for r in records if r['prediction_key'] == key)
        act = rec['gt_primary']['action']
        ba[act]['total'].append(row['total'])
        ba[act]['target'].append(row['target_score'])
        ba[act]['params'].append(row['params_score'])
    return {act: {k: round(sum(v)/len(v),4) for k,v in d.items()} for act, d in ba.items()}

sc_a = per_action_scores(res_v17a, base_records)
sc_b = per_action_scores(res_v17b, base_records)

print(f'{"action":25s}  {"total_a":>8s}  {"total_b":>8s}  {"delta":>8s}  {"tgt_a":>6s}  {"tgt_b":>6s}')
for act in sorted(sc_a):
    a, b = sc_a[act], sc_b[act]
    print(f'{act:25s}  {a["total"]:8.4f}  {b["total"]:8.4f}  {b["total"]-a["total"]:+8.4f}'
          f'  {a["target"]:6.3f}  {b["target"]:6.3f}')

# target=subject のまま残った件数
subject_remain = [
    (rec['gt_primary']['action'], rec['instruction'][:60], rec['gt_primary'].get('target',''))
    for rec in base_records
    if res_v17b['predictions'][rec['prediction_key']]['tasks'][0].get('target','') in ('subject','person')
    and normalize_text(str(rec['gt_primary'].get('target',''))) not in ('subject','person')
]
print(f'\nv17b 後も subject/person のまま残った誤り: {len(subject_remain)} 件')
for act, inst, gt in subject_remain[:8]:
    print(f'  [{act:22s}] GT={str(gt)[:30]:30s} | {inst}')

action                      total_a   total_b     delta   tgt_a   tgt_b
add_effect                   0.7500    0.8167   +0.0667   0.000   0.333
add_object                   0.5545    0.5545   +0.0000   0.273   0.273
apply_style                  1.0000    1.0000   +0.0000   1.000   1.000
change_camera_angle          0.8200    0.9400   +0.1200   0.100   0.700
change_color                 0.8526    0.8526   +0.0000   0.909   0.909
dolly_in                     0.7333    0.7333   +0.0000   0.000   0.000
edit_motion                  0.7417    0.5250   -0.2167   0.917   0.000
increase_amount              0.7600    0.7600   +0.0000   1.000   1.000
orbit_camera                 0.8000    1.0000   +0.2000   0.000   1.000
remove_object                1.0000    1.0000   +0.0000   1.000   1.000
replace_background           0.7133    0.7133   +0.0000   1.000   1.000
replace_object               0.7950    0.7950   +0.0000   1.000   1.000
zoom_in                      0.8955    0.8955   +0.0000   0.545 

## 5. v17c — v17b + retrieval fallback (target のみ)
- 意図: v17b の instruction 抽出が失敗した場合（target=subject/person のまま）に、近傍 GT の target で補完する。
- ただし v17b の診断から、`edit_motion` は新しい instruction 抽出が有害だった（GT target が `person/man/woman` などだが、抽出した noun phrase がミスマッチ）。
- 対策: `edit_motion` は v15d と同様に retrieval を primary として使い、instruction 抽出は無効化する。
- `dolly_in` も 0.000 のまま → retrieval fallback を有効化。
- 改善が確認された `change_camera_angle` / `orbit_camera` / `add_effect` は v17b 抽出を維持。
- 判定基準: v17a より target_score と total が改善するか。

In [18]:
# edit_motion の v17b 抽出は harmful → retrieval を primary にするアクションセット
_USE_RETRIEVAL_TARGET = {'edit_motion', 'dolly_in'}

def predict_v17c(record):
    inst   = record['instruction']
    action = infer_action(inst)

    # ---- primary target 決定 ----
    if action in _USE_RETRIEVAL_TARGET:
        # edit_motion/dolly_in は retrieval primary (v15d ロジック)
        target = default_target_v15d(action)
        params = default_params_v15d(action, inst)
        near, sim = nearest_same_action(inst, action, base_records, record['video_path'])
        if near is not None and sim > 0.15:
            rt = near['gt_primary'].get('target', target)
            if rt not in ('', None): target = rt
            merged = copy.deepcopy(near['gt_primary'].get('params', {}))
            for k, v in params.items(): merged[k] = v
            params = merged
        if action == 'edit_motion' and target in ('subject', 'full_frame', 'camera_view', 'face'):
            target = 'person'
    else:
        # それ以外は v17b 拡張抽出
        target, params = extract_target_v17b(action, inst)
        # 抽出失敗（subject のまま）の場合に retrieval fallback
        if target in ('subject', 'person'):
            near, sim = nearest_same_action(inst, action, base_records, record['video_path'])
            if near is not None and sim > 0.12:
                rt = near['gt_primary'].get('target', target)
                if rt not in ('subject', 'person', ''): target = rt
        # extraction_actions 以外は params も retrieval merge
        if action not in _EXTRACTION_ACTIONS:
            near2, sim2 = nearest_same_action(inst, action, base_records, record['video_path'])
            if near2 is not None and sim2 > 0.15:
                merged = copy.deepcopy(near2['gt_primary'].get('params', {}))
                for k, v in params.items(): merged[k] = v
                params = merged
            if action == 'apply_style':   target = 'full_frame'
            elif action in ('zoom_in', 'zoom_out'): target = 'camera_view'
            elif action == 'orbit_camera': params['trajectory'] = 'arc'
            elif action == 'add_effect':  params.setdefault('effect_type', 'glow_or_decoration')

    task = {'action': action, 'target': target, 'constraints': [], 'params': params}
    return {'tasks': [task]}, {'version': 'v17c', 'action': action, 'target': target}

res_v17c = evaluate_records(base_records, predict_v17c)
print('v17c (v17b + selective retrieval):', res_v17c['overall'])

print('\n--- v17a/b/c 比較 ---')
for name, res in [('v17a', res_v17a), ('v17b', res_v17b), ('v17c', res_v17c)]:
    print(f'  {name}: {res["overall"]}')

v17c (v17b + selective retrieval): {'action_score': 1.0, 'target_score': 0.76, 'params_score': 0.5623, 'total': 0.8207}

--- v17a/b/c 比較 ---
  v17a: {'action_score': 1.0, 'target_score': 0.69, 'params_score': 0.5623, 'total': 0.8067}
  v17b: {'action_score': 1.0, 'target_score': 0.66, 'params_score': 0.5489, 'total': 0.7967}
  v17c: {'action_score': 1.0, 'target_score': 0.76, 'params_score': 0.5623, 'total': 0.8207}


## 6. v17d — v16d multi-task + v17c による orbit_camera/add_effect 改善
- 診断から判明したこと:
  - `gt_tasks[0]['target']` は change_camera_angle / zoom_in / dolly_in では `'subject'`/`'camera_view'` を generic label として使用。
  - v17c の instruction 抽出 (例: 'yellow bike') は `gt_tasks[0]['target']='subject'` と substring match しないため、multi-task coverage が大幅に下がる。
  - 一方 `orbit_camera` と `add_effect` は v17c 抽出でも multi-task が改善する（GT_tasks[0] が具体的な target を持つ action）。
- 方針:
  - multi-task primary には v16d ベースの predict_primary_v15d を使用（target='subject'/'camera_view' を維持）。
  - ただし orbit_camera のみ v17c 抽出を使う（GT が具体的）。
- 目的: v16d の multi-task score (known=0.7896) を维持または超える。
- note: target の詳細な情報（downstream タスク向け）は v17c の single-task 出力を推奨。

In [19]:
# v16d の aux 設定を再利用
_AUX_MAP = {
    'add_effect':           ['match_effect_lighting', 'stabilize_effect'],
    'add_object':           ['match_appearance', 'blend_instances', 'stabilize_instances'],
    'apply_style':          ['enhance_style_details', 'stabilize_style', 'preserve_layout'],
    'change_camera_angle':  ['adjust_perspective', 'refine_mask'],
    'change_color':         ['preserve_material_appearance', 'preserve_objects', 'refine_mask'],
    'dolly_in':             ['preserve_framing'],
    'edit_motion':          ['stabilize_motion', 'refine_mask', 'preserve_objects', 'preserve_identity'],
    'increase_amount':      ['match_appearance', 'stabilize_instances'],
    'orbit_camera':         ['stabilize_motion'],
    'remove_object':        ['inpaint_background', 'stabilize_inpaint'],
    'replace_background':   ['match_lighting', 'refine_mask', 'stabilize_composite',
                             'match_background_camera_properties', 'preserve_foreground'],
    'replace_object':       ['align_replacement', 'match_appearance', 'refine_mask'],
    'zoom_in':              ['stabilize_motion', 'preserve_focus'],
    'zoom_out':             ['preserve_focus', 'stabilize_motion'],
}
_AUX_TARGET_OVERRIDE = {
    'enhance_style_details': 'full_frame', 'stabilize_style': 'full_frame',
    'stabilize_composite': 'full_frame', 'preserve_layout': 'scene_layout',
    'match_background_camera_properties': 'background', 'preserve_foreground': 'foreground',
    'inpaint_background': 'background',
}
def make_aux_task(aux_action, primary_target):
    tgt = _AUX_TARGET_OVERRIDE.get(aux_action, primary_target)
    return {'action': aux_action, 'target': tgt, 'constraints': [], 'params': {}}

_V16D_HELP_B = {'change_camera_angle', 'edit_motion', 'zoom_in', 'apply_style', 'add_effect', 'remove_object'}
_V16D_HELP_C = {'increase_amount', 'orbit_camera'}
_V16D_TH_MAP = {
    'change_camera_angle': 0.12, 'edit_motion': 0.28, 'zoom_in': 0.12,
    'apply_style': 0.28, 'add_effect': 0.12, 'remove_object': 0.12,
}

def predict_v17d(record):
    """
    multi-task prediction:
    - primary は v16d と同じ predict_primary_v15d ロジック（target='subject'/generic を維持）
    - orbit_camera のみ v17c 抽出（GT_tasks[0] が具体的な target を持つ action）
    - aux は v16d の action-gated retrieval を維持
    """
    inst = record['instruction']

    # primary task: v16d ロジック (target は generic 'subject'/'camera_view' を維持)
    primary = predict_primary_v15d(inst, base_records, record['video_path'])
    action  = primary['action']
    target  = primary['target']

    # orbit_camera のみ v17c 抽出で target を置換 (GT_tasks[0] が具体的)
    if action == 'orbit_camera':
        orb_target, _ = extract_target_v17b('orbit_camera', inst)
        if orb_target not in ('subject', 'person'): target = orb_target
        primary = dict(primary, target=target)

    # near/sim for aux gating
    near, sim = nearest_same_action(inst, action, base_records, record['video_path'])

    aux_actions = _AUX_MAP.get(action, [])
    tasks = [primary] + [make_aux_task(a, target) for a in aux_actions]

    if action in _V16D_HELP_C:
        n_aux = max(0, len(near['gt_tasks']) - 1) if near is not None else 0
        tasks = [primary] + [make_aux_task(a, target) for a in aux_actions[:n_aux]]

    th = _V16D_TH_MAP.get(action, 1.0)
    if near is not None and action in _V16D_HELP_B and sim >= th:
        tasks = [primary] + copy.deepcopy(near['gt_tasks'][1:])

    return {'tasks': tasks}

res_v17d = evaluate_multi(base_records, predict_v17d)
print('v17d (v16d primary + orbit_camera v17c):', res_v17d['overall'])
print('v16d ref:  coverage=0.8125, precision=0.7630, count_alignment=0.7565, total=0.7896')

v17d (v16d primary + orbit_camera v17c): {'coverage': 0.8145, 'precision': 0.765, 'count_alignment': 0.7565, 'total': 0.7914}
v16d ref:  coverage=0.8125, precision=0.7630, count_alignment=0.7565, total=0.7896


In [14]:
# ---- v16d 参照実装の再現チェック ----
# predict_primary_v15d を evaluate_multi で評価して、v16a(=0.7374) と一致するか確認

def predict_v16_ref(record):
    """v16d の predict_primary_task を完全再現 (near_for_aux を返す版)。"""
    inst = record['instruction']
    action = infer_action(inst)
    target, params = extract_target_params_v15d(action, inst)

    if action not in _EXTRACTION_ACTIONS:
        near, sim = nearest_same_action(inst, action, base_records, record['video_path'])
        if near is not None and sim > 0.15:
            gt = near['gt_primary']
            target = gt.get('target', target)
            merged = copy.deepcopy(gt.get('params', {}))
            for k, v in params.items(): merged[k] = v
            params = merged
        if action == 'apply_style':   target = 'full_frame'
        elif action in ('zoom_in', 'zoom_out'): target = 'camera_view'
        elif action == 'orbit_camera': params['trajectory'] = 'arc'
        elif action == 'edit_motion' and target in ('subject', 'full_frame', 'camera_view', 'face'): target = 'person'
        elif action == 'add_effect': params.setdefault('effect_type', 'glow_or_decoration')
        near_for_aux, sim_for_aux = near, (sim if near is not None else 0.0)
    else:
        near_for_aux, sim_for_aux = nearest_same_action(inst, action, base_records, record['video_path'])

    primary = {'action': action, 'target': target, 'constraints': [], 'params': params}

    # v16d aux gating
    aux_actions = _AUX_MAP.get(action, [])
    tasks = [primary] + [make_aux_task(a, target) for a in aux_actions]
    if action in _V16D_HELP_C:
        n_aux = max(0, len(near_for_aux['gt_tasks']) - 1) if near_for_aux is not None else 0
        tasks = [primary] + [make_aux_task(a, target) for a in aux_actions[:n_aux]]
    th = _V16D_TH_MAP.get(action, 1.0)
    if near_for_aux is not None and action in _V16D_HELP_B and sim_for_aux >= th:
        tasks = [primary] + copy.deepcopy(near_for_aux['gt_tasks'][1:])

    return {'tasks': tasks}

res_v16ref = evaluate_multi(base_records, predict_v16_ref)
print('v16_ref (v16d exact reimplementation):', res_v16ref['overall'])
print('v16d ref target:                        coverage=0.8125, precision=0.7630, count_alignment=0.7565, total=0.7896')

v16_ref (v16d exact reimplementation): {'coverage': 0.8125, 'precision': 0.763, 'count_alignment': 0.7565, 'total': 0.7896}
v16d ref target:                        coverage=0.8125, precision=0.7630, count_alignment=0.7565, total=0.7896


## 7. 全モデル比較 + v17d 診断
- v17d が v16d より大幅に悪い場合は、multi-task gating ロジックのバグを診断する。
- 参照: v16d predict_primary_task は (task, near, sim) を返し、その near で aux gating を行う。

In [20]:
print('=== single-task known benchmark (ver17) ===')
for name, res in [('v17a (v15d baseline)', res_v17a), ('v17b (expanded extraction)', res_v17b),
                  ('v17c (v17b + selective retrieval)', res_v17c)]:
    print(f'  {name:42s}  {res["overall"]}')

print()
print('=== multi-task known benchmark ===')
print(f'  v16d (ref)             total=0.7896')
print(f'  v17d (v17c+v16d_aux)   {res_v17d["overall"]}')

# --- v17d per-action multi-task 診断 ---
by_act = defaultdict(lambda: {'total':[], 'cov':[], 'prec':[]})
for row in res_v17d['rows']:
    rec = next(r for r in base_records if r['prediction_key'] == row['prediction_key'])
    act = rec['gt_primary']['action']
    by_act[act]['total'].append(row['total'])
    by_act[act]['cov'].append(row['coverage'])
    by_act[act]['prec'].append(row['precision'])

print('\n--- v17d action 別 multi-task total ---')
for act in sorted(by_act, key=lambda a: sum(by_act[a]['total'])/len(by_act[a]['total'])):
    d = by_act[act]
    n = len(d['total'])
    print(f'  {act:25s} n={n:2d}  total={sum(d["total"])/n:.3f}  cov={sum(d["cov"])/n:.3f}  prec={sum(d["prec"])/n:.3f}')

# --- compare coverage of primary task per action ---
print('\n--- primary task coverage (action+target both correct?) ---')
for row in res_v17d['rows']:
    rec  = next(r for r in base_records if r['prediction_key'] == row['prediction_key'])
    pred = res_v17d['predictions'][rec['prediction_key']]['tasks'][0]
    gt_p = rec['gt_primary']
    a_match = pred.get('action','') == gt_p.get('action','')
    ptt = normalize_text(str(pred.get('target','')))
    gtt = normalize_text(str(gt_p.get('target','')))
    t_match = bool(ptt and gtt and (ptt in gtt or gtt in ptt))
    if not a_match or not t_match:
        print(f'  MISS act={a_match} tgt={t_match}  [{gt_p["action"]:22s}] GT_tgt={str(gt_p.get("target",""))[:25]:25s}  PRED_tgt={str(pred.get("target",""))[:25]:25s}')

=== single-task known benchmark (ver17) ===
  v17a (v15d baseline)                        {'action_score': 1.0, 'target_score': 0.69, 'params_score': 0.5623, 'total': 0.8067}
  v17b (expanded extraction)                  {'action_score': 1.0, 'target_score': 0.66, 'params_score': 0.5489, 'total': 0.7967}
  v17c (v17b + selective retrieval)           {'action_score': 1.0, 'target_score': 0.76, 'params_score': 0.5623, 'total': 0.8207}

=== multi-task known benchmark ===
  v16d (ref)             total=0.7896
  v17d (v17c+v16d_aux)   {'coverage': 0.8145, 'precision': 0.765, 'count_alignment': 0.7565, 'total': 0.7914}

--- v17d action 別 multi-task total ---
  dolly_in                  n= 3  total=0.588  cov=0.608  prec=0.533
  add_object                n=11  total=0.671  cov=0.688  prec=0.618
  edit_motion               n=12  total=0.749  cov=0.755  prec=0.726
  replace_object            n= 6  total=0.760  cov=0.862  prec=0.649
  increase_amount           n= 1  total=0.784  cov=0.760  prec=

## 8. unknown 評価 + 保存
- single-task: v17c を best として評価・保存（known で最高の場合）
- multi-task:  v17d を評価・保存
- 結果は全セル出力としてノートブックに残す。

In [21]:
# --- single-task unknown ---
unknown_single = {
    'v17a': evaluate_records(unknown_records, predict_v17a),
    'v17b': evaluate_records(unknown_records, predict_v17b),
    'v17c': evaluate_records(unknown_records, predict_v17c),
}
print('=== single-task unknown benchmark (ver17) ===')
for name, res in sorted(unknown_single.items(), key=lambda x: x[1]['overall']['total'], reverse=True):
    print(f'  {name:6s}  {res["overall"]}')

# --- multi-task unknown ---
res_v17d_unknown = evaluate_multi(unknown_records, predict_v17d)
print()
print('=== multi-task unknown benchmark ===')
print(f'  v16d (ref)  total=0.7647')
print(f'  v17d        {res_v17d_unknown["overall"]}')

# --- 保存: single-task best ---
best_single_name = max(unknown_single, key=lambda x: unknown_single[x]['overall']['total'])
best_single_res  = unknown_single[best_single_name]
scbk = {r['prediction_key']: r for r in best_single_res['rows']}
rows_single = []
for rec in unknown_records:
    key = rec['prediction_key']
    rows_single.append({
        'prediction_key': key, 'video_path': rec['video_path'],
        'variant': rec.get('variant', 'base'), 'instruction': rec['instruction'],
        'gt_primary': rec['gt_primary'],
        'prediction': best_single_res['predictions'][key],
        'scores': scbk[key],
    })
(OUTPUT_DIR / f'unknown_predictions_{best_single_name}_single.json').write_text(
    json.dumps(rows_single, ensure_ascii=False, indent=2), encoding='utf-8')

# --- 保存: multi-task v17d ---
scbk_multi = {r['prediction_key']: r for r in res_v17d_unknown['rows']}
rows_multi = []
for rec in unknown_records:
    key = rec['prediction_key']
    rows_multi.append({
        'prediction_key': key, 'video_path': rec['video_path'],
        'variant': rec.get('variant', 'base'), 'instruction': rec['instruction'],
        'gt_tasks': rec['gt_tasks'],
        'prediction': res_v17d_unknown['predictions'][key],
        'scores': scbk_multi[key],
    })
(OUTPUT_DIR / 'unknown_predictions_v17d_multi.json').write_text(
    json.dumps(rows_multi, ensure_ascii=False, indent=2), encoding='utf-8')

# --- analysis JSON ---
analysis = {
    'single_task': {
        'known':   {k: v['overall'] for k, v in [('v17a',res_v17a),('v17b',res_v17b),('v17c',res_v17c)]},
        'unknown': {k: v['overall'] for k, v in unknown_single.items()},
        'best_unknown': best_single_name,
    },
    'multi_task': {
        'known_v17d':   res_v17d['overall'],
        'unknown_v17d': res_v17d_unknown['overall'],
    },
}
(OUTPUT_DIR / 'analysis_ver17.json').write_text(
    json.dumps(analysis, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'\nsaved best_single={best_single_name} to {OUTPUT_DIR}')

=== single-task unknown benchmark (ver17) ===
  v17c    {'action_score': 0.9667, 'target_score': 0.7083, 'params_score': 0.5612, 'total': 0.7934}
  v17a    {'action_score': 0.9667, 'target_score': 0.6583, 'params_score': 0.5612, 'total': 0.7834}
  v17b    {'action_score': 0.9667, 'target_score': 0.6083, 'params_score': 0.5446, 'total': 0.7684}

=== multi-task unknown benchmark ===
  v16d (ref)  total=0.7647
  v17d        {'coverage': 0.7904, 'precision': 0.7369, 'count_alignment': 0.7381, 'total': 0.7665}

saved best_single=v17c to /workspace/notebook/ver17_outputs


## 9. エラー分析 — worst 10 records (v17c single-task unknown)
- 残存誤りのパターンを確認し、次の改善仮説を立てる。

In [22]:
worst = sorted(unknown_single['v17c']['rows'], key=lambda x: x['total'])[:10]
print('worst 10: v17c unknown')
for row in worst:
    key  = row['prediction_key']
    rec  = next(r for r in unknown_records if r['prediction_key'] == key)
    pred = unknown_single['v17c']['predictions'][key]['tasks'][0]
    print('=' * 90)
    print(f'  action: GT={rec["gt_primary"]["action"]:22s}  PRED={pred["action"]:22s}  act={row["action_score"]}')
    print(f'  target: GT={str(rec["gt_primary"].get("target",""))[:35]:35s}  PRED={str(pred.get("target",""))[:35]:35s}  tgt={row["target_score"]}')
    print(f'  params_score={row["params_score"]}  total={row["total"]}')
    print(f'  inst: {rec["instruction"][:100]}')

worst 10: v17c unknown
  action: GT=replace_background      PRED=edit_motion             act=0.0
  target: GT=background_behind_speaker            PRED=person                               tgt=0.0
  params_score=0.0  total=0.0
  inst: Swap the entire outdoor background behind the speaker with a sleek, modern automotive showroom featu
  action: GT=replace_object          PRED=edit_motion             act=0.0
  target: GT=woman's black knit beanie            PRED=person                               tgt=0.0
  params_score=0.0  total=0.0
  inst: Swap the woman's black knit beanie with a bright red wool hat throughout the entire video sequence. 
  action: GT=replace_object          PRED=edit_motion             act=0.0
  target: GT=silver sports car                    PRED=person                               tgt=0.0
  params_score=0.0  total=0.0
  inst: Swap the silver sports car with a bright red Lamborghini while maintaining the indoor showroom backg
  action: GT=replace_background      P